# Wildfire Untrained Baseline (Kaggle)

Use this notebook only to measure **untrained LLM performance**.

Cell order:
1. Run Cell 1 first each session (loads secrets + BASE).
2. Cells 2-5 are safe to re-run.

In [ ]:
# Cell 1 (run first each session): secrets + constants
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN

REPO = "jhawaritvik/Scaler-hackathon"
REPO_URL = f"https://github.com/{REPO}.git"
BASE = "/kaggle/working/wildfire-env"

print("Configured BASE:", BASE)
print("Tokens loaded: GITHUB_TOKEN + HF_TOKEN")

In [ ]:
# Cell 2: clone/update + install deps (safe to re-run)
import os
import glob
import subprocess
import sys

if not os.path.exists(BASE):
    subprocess.run(["git", "clone", f"https://{GITHUB_TOKEN}@github.com/{REPO}.git", BASE], check=True)

os.chdir(BASE)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "unsloth", "unsloth-zoo", "bitsandbytes", "xgrammar", "transformers", "trl"], check=True)

nvidia_lib_dirs = glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/*/lib")
if nvidia_lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

print("Install complete. cwd:", os.getcwd())

In [ ]:
# Cell 3: preflight for baseline path
import subprocess
import sys

subprocess.run(["openenv", "validate", "."], check=True)
subprocess.run([sys.executable, "smoke_test.py"], check=True)
print("openenv validate passed")

In [ ]:
# Cell 4: run untrained eval and save artifact
import os
import subprocess
import sys

os.makedirs("submission_artifacts", exist_ok=True)
subprocess.run([
    sys.executable,
    "eval_policy.py",
    "--untrained",
    "--output",
    "submission_artifacts/eval_untrained.json",
], check=True)

print("Wrote submission_artifacts/eval_untrained.json")

In [ ]:
# Cell 5: optional push of untrained artifact to GitHub + quick view
import json
import subprocess

with open("submission_artifacts/eval_untrained.json", "r", encoding="utf-8") as f:
    untrained = json.load(f)

print("overall_mean:", untrained.get("overall_mean"))
print(json.dumps(untrained, indent=2)[:2000])

AUTO_PUSH = False  # set True when you want to push from this notebook
if AUTO_PUSH:
    subprocess.run(["git", "add", "submission_artifacts/eval_untrained.json"], check=True)
    subprocess.run(["git", "commit", "-m", "Add untrained baseline eval artifact"], check=True)
    subprocess.run(["git", "push", "origin", "main"], check=True)
    print("Pushed eval_untrained.json to main")